In [1]:
import pandas as pd

files = {
#    "XXX": "Out_of_sample_ABX3_RFR_predictions.xlsx",   # XXX is name of the gregression
}

property_cols = [
    "Predicted Efficiency (%)",
    "Predicted Open circuit voltage (V)",
    "Predicted Short circuit current density (A/m²)",
    "Predicted Fill factor",
    "Predicted Band gap (eV)"
]

target_Eg = 1.34
top_n = 10

# Read each file only once
model_data = {
    model_name: pd.read_excel(file_name)
    for model_name, file_name in files.items()
}

all_top = []

# Select top compounds from each model
for model_name, df in model_data.items():

    required_cols = ["compound"] + property_cols
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{model_name} is missing the following columns: {missing_cols}"
        )

    for prop in property_cols:

        # For band gap, select compounds closest to target Eg
        if prop == "Predicted Band gap (eV)":
            selected = (
                df.assign(Eg_error=(df[prop] - target_Eg).abs())
                  .sort_values(by=["Eg_error", prop])
                  .head(top_n)
                  .copy()
            )

        # For other properties, select highest predicted values
        else:
            selected = (
                df.sort_values(by=prop, ascending=False)
                  .head(top_n)
                  .copy()
            )

        selected["Model"] = model_name
        selected["Property"] = prop
        selected["Rank"] = range(1, len(selected) + 1)

        all_top.append(selected)

top_all_models = pd.concat(all_top, ignore_index=True)

# Count model agreement
consistent = (
    top_all_models
    .groupby(["Property", "compound"])
    .agg(
        Models=("Model", lambda x: ", ".join(sorted(x.unique()))),
        Model_count=("Model", "nunique"),
        Best_rank=("Rank", "min"),
        Mean_rank=("Rank", "mean")
    )
    .reset_index()
)

# Sort by property, model agreement, and rank
consistent = consistent.sort_values(
    by=["Property", "Model_count", "Best_rank", "Mean_rank"],
    ascending=[True, False, True, True]
)

# Compounds selected by exactly 2, 3, 4, or all models
consistent_2models = consistent[consistent["Model_count"] == 2].copy()
consistent_3models = consistent[consistent["Model_count"] == 3].copy()
consistent_4models = consistent[consistent["Model_count"] == 4].copy()

number_of_models = len(files)

consistent_all_models = consistent[
    consistent["Model_count"] == number_of_models
].copy()

# Save results
output_file = "Consistent_top10_compounds_across_models.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    top_all_models.to_excel(
        writer,
        sheet_name="All_top10_by_model",
        index=False
    )

    consistent.to_excel(
        writer,
        sheet_name="Agreement_summary",
        index=False
    )

    consistent_2models.to_excel(
        writer,
        sheet_name="Exactly_2_models",
        index=False
    )

    consistent_3models.to_excel(
        writer,
        sheet_name="Exactly_3_models",
        index=False
    )

    consistent_4models.to_excel(
        writer,
        sheet_name="Exactly_4_models",
        index=False
    )

    consistent_all_models.to_excel(
        writer,
        sheet_name="All_models",
        index=False
    )

print(f"Saved as {output_file}")

Saved as Consistent_top10_compounds_across_models.xlsx
